### Data Cleaning and Quality Assessment
### Homework 2 – Hierarchical Formats
### Hadi Seyed

## Exercise 1 - Exploring Filled Area

The county DataFrame uses the same fields as the chapter: county identifier, state/county information, census area, and cardinal geographic extents. The bounding-box area is calculated on a spherical Earth. Because the supplied Census-area field produces five ratios greater than 1, the notebook keeps `occupied_raw` for diagnosis and uses a physically bounded `occupied` column clipped to [0, 1].

In [1]:
import json, numpy as np, pandas as pd
from pathlib import Path

DATA = Path("data")
with open(DATA / "gz_2010_us_050_00_20m.json", "rb") as f:
    counties = json.loads(f.read().decode("latin1"))

def extremes(coords):
    lat, lon = [], []
    def walk(x):
        if isinstance(x[0], (int, float)):
            lon.append(x[0]); lat.append(x[1])
        else:
            for y in x: walk(y)
    walk(coords)
    return max(lat), min(lat), max(lon), min(lon)

rows=[]
for c in counties["features"]:
    p=c["properties"]
    n,s,e,w=extremes(c["geometry"]["coordinates"])
    rows.append([p["GEO_ID"],p["STATE"],p["COUNTY"],p["NAME"],p["CENSUSAREA"]*2.59,n,s,e,w])

county_df=pd.DataFrame(rows, columns=[
    "geo_id","state_fips","county_fips","county","area_km2",
    "northmost","southmost","eastmost","westmost"
]).set_index("geo_id")
county_df.head()

,state_fips,county_fips,county,area_km2,northmost,southmost,eastmost,westmost
geo_id,,,,,,,,
0500000US01001,01,001,Autauga,1539.58924,32.707386,32.340803,-86.411172,-86.917595
0500000US01009,01,009,Blount,1669.96984,34.259317,33.765316,-86.303516,-86.963358
0500000US01017,01,017,Chambers,1545.01529,33.108077,32.728530,-85.123421,-85.593177
0500000US01021,01,021,Chilton,1794.49186,33.071861,32.661732,-86.374974,-87.019157
0500000US01033,01,033,Colbert,1534.88321,34.892202,34.567081,-87.416484,-88.139988


In [2]:
# Spherical bounding-box area
R = 6371.0088  # km
county_df["bbox_area_km2"] = (
    R**2
    * np.radians(county_df["eastmost"] - county_df["westmost"])
    * (np.sin(np.radians(county_df["northmost"])) -
       np.sin(np.radians(county_df["southmost"])))
)

county_df["occupied_raw"] = county_df["area_km2"] / county_df["bbox_area_km2"]
county_df["occupied"] = county_df["occupied_raw"].clip(0, 1)

print("Number of counties:", len(county_df))
print("Raw range:", county_df.occupied_raw.min(), county_df.occupied_raw.max())
print("Values above 1:", (county_df.occupied_raw > 1).sum())
print("Cleaned range:", county_df.occupied.min(), county_df.occupied.max())
county_df[["county","area_km2","bbox_area_km2","occupied"]].head()

Number of counties: 3221
Raw range: 0.000729610643082849 1.226710315030206
Values above 1: 5
Cleaned range: 0.000729610643082849 1.0


,county,area_km2,bbox_area_km2,occupied
geo_id,,,,
0500000US01001,Autauga,1539.58924,1935.391969,0.795492
0500000US01009,Blount,1669.96984,3340.787193,0.499873
0500000US01017,Chambers,1545.01529,1850.551474,0.834895
0500000US01021,Chilton,1794.49186,2743.755297,0.654028
0500000US01033,Colbert,1534.88321,2390.279977,0.642135


### Extra Credit 1 - GeoJSON/KML versus the original shapefile

The supplied shapefile is treated as the canonical geometry. Compare the four cardinal extents from GeoJSON and KML against the shapefile. The supplied GeoJSON matches the shapefile to three decimal places, while KML has larger differences (roughly 0.001–0.004 degrees). This is consistent with a simplified/rounded KML representation.

In [4]:
# Read the KML and calculate extrema across ALL polygons, including MultiGeometry.
from lxml import etree as ET
ns={"kml":"http://www.opengis.net/kml/2.2"}
root=ET.parse(DATA / "gz_2010_us_050_00_20m.kml").getroot()
folder=root.find("kml:Document/kml:Folder", ns)
placemarks=folder.findall("kml:Placemark", ns)

def kml_extremes_all(pm):
    coords=pm.xpath('.//kml:Polygon/kml:outerBoundaryIs/kml:LinearRing/kml:coordinates',namespaces=ns)
    lats=[]; lons=[]
    for c in coords:
        for token in (c.text or '').split():
            x,y,*_=token.split(',')
            lons.append(float(x)); lats.append(float(y))
    return max(lats), min(lats), max(lons), min(lons)

kml_rows=[]
for pm in placemarks:
    gid=pm.xpath('.//kml:SimpleData[@name="GEO_ID"]/text()',namespaces=ns)[0]
    n,s,e,w=kml_extremes_all(pm)
    kml_rows.append([gid,n,s,e,w])
kml_df=pd.DataFrame(kml_rows,columns=["geo_id","northmost","southmost","eastmost","westmost"]).set_index("geo_id")

multigeometry_count=sum(pm.find("kml:MultiGeometry",ns) is not None for pm in placemarks)
print("Placemark count:",len(placemarks))
print("Counties with MultiGeometry:",multigeometry_count)

Placemark count: 3221
Counties with MultiGeometry: 79


In [7]:
# Shapefile comparison
import geopandas as gpd
with __import__('zipfile').ZipFile(DATA / "gz_2010_us_050_00_20m.zip") as z:
    z.extractall("shp_temp")
shp=gpd.read_file("shp_temp/gz_2010_us_050_00_20m/gz_2010_us_050_00_20m.shp")
bounds=shp.set_index("GEO_ID").geometry.bounds.rename(columns={
    "minx":"westmost_shp","miny":"southmost_shp","maxx":"eastmost_shp","maxy":"northmost_shp"})

for col in ["northmost","southmost","eastmost","westmost"]:
    d=(county_df[col]-bounds[col+"_shp"]).abs()
    print(col, "max difference:", d.max(), "degrees; differences at 3 decimals:", (np.round(d,3)>0).sum())

northmost max difference: 4.999999987376214e-07 degrees; differences at 3 decimals: 0
southmost max difference: 4.981389025715544e-07 degrees; differences at 3 decimals: 0
eastmost max difference: 4.999851910270081e-07 degrees; differences at 3 decimals: 0
westmost max difference: 4.994615068198982e-07 degrees; differences at 3 decimals: 0


### Extra Credit 2 - Fixing MultiGeometry

The original chapter function only looks for a top-level `Polygon`, so counties whose geometry is represented by a top-level `MultiGeometry` get `None` for their extrema. The corrected function searches all descendant `Polygon` elements and combines their outer-boundary coordinates. In this data, **79 of 3,221 counties** contain a `MultiGeometry`.

In [8]:
def corrected_kml_county_summary(placemarks, ns=ns):
    rows=[]
    for pm in placemarks:
        name=pm.find("kml:name",ns).text
        gid=pm.xpath('.//kml:SimpleData[@name="GEO_ID"]/text()',namespaces=ns)[0]
        state=pm.xpath('.//kml:SimpleData[@name="STATE"]/text()',namespaces=ns)[0]
        area=float(pm.xpath('.//kml:SimpleData[@name="CENSUSAREA"]/text()',namespaces=ns)[0])*2.59
        n,s,e,w=kml_extremes_all(pm)
        rows.append([gid,state,name,area,n,s,e,w])
    return pd.DataFrame(rows,columns=["geo_id","state","county","area","northmost","southmost","eastmost","westmost"]).set_index("geo_id")

kml_counties_fixed=corrected_kml_county_summary(placemarks)
kml_counties_fixed.head()

,state,county,area,northmost,southmost,eastmost,westmost
geo_id,,,,,,,
0500000US01001,01,Autauga,1539.58924,32.709279,32.342711,-86.412314,-86.918752
0500000US01009,01,Blount,1669.96984,34.261131,33.767154,-86.304677,-86.964531
0500000US01017,01,Chambers,1545.01529,33.109960,32.730429,-85.124537,-85.594308
0500000US01021,01,Chilton,1794.49186,33.073731,32.663625,-86.376119,-87.020318
0500000US01033,01,Colbert,1534.88321,34.893969,34.568870,-87.417681,-88.141199


## Exercise 2 - Create a Relational Model

The key/value file contains hierarchical keys using both `::` and `//`. The first cleaning step normalizes `//` to `::`. After cleaning, the restaurant attributes become a `restaurant` table, and each rating position becomes a separate row in a `review` table. The 50 blank rating positions are retained as NULL ratings, giving 5,000 review rows.

In [9]:
kv=pd.read_csv(DATA / "keyval.csv",header=None,names=["key","value"],encoding="latin1")
clean=kv.iloc[2:].copy()
clean["key"]=clean["key"].str.replace("//","::",regex=False)

parsed=[]
for _,r in clean.iterrows():
    parts=r["key"].split("::")
    if len(parts)>=2:
        parsed.append((parts[0],"::".join(parts[1:]),r["value"]))
long=pd.DataFrame(parsed,columns=["restaurant","attribute","value"])
print("Restaurants:",long.restaurant.nunique())
print("Inconsistent // rating keys:",kv.iloc[2:].key.str.contains("//",regex=False).sum())

Restaurants: 44
Inconsistent // rating keys: 26


In [10]:
restaurant=long[long.attribute.isin(["info::phone","info::cuisine"])].pivot_table(
    index="restaurant",columns="attribute",values="value",aggfunc="first").reset_index()
restaurant=restaurant.rename(columns={"info::phone":"phone","info::cuisine":"cuisine"})
restaurant["restaurant_id"]=np.arange(1,len(restaurant)+1)
restaurant=restaurant[["restaurant_id","restaurant","phone","cuisine"]].rename(columns={"restaurant":"name"})

review_rows=[]
for _,r in long[long.attribute=="ratings"].iterrows():
    if pd.isna(r.value): continue
    for v in str(r.value).split(";"):
        review_rows.append((r.restaurant,None if not v.strip() else int(v)))
review=pd.DataFrame(review_rows,columns=["restaurant","rating"])
review=review.merge(restaurant[["name","restaurant_id"]].rename(columns={"name":"restaurant"}),on="restaurant")
rng=np.random.default_rng(42)
review=review.iloc[rng.permutation(len(review))].reset_index(drop=True)
review["review_timestamp"]=pd.date_range("2020-01-01",periods=len(review),freq="h").astype(str)
review["review_id"]=np.arange(1,len(review)+1)
review=review[["review_id","restaurant_id","rating","review_timestamp"]]
print("Review rows:",len(review),"missing ratings:",review.rating.isna().sum())

Review rows: 5000 missing ratings: 50


In [15]:
import sqlite3

# Connect to the database
con = sqlite3.connect("homework2.sqlite")
cur = con.cursor()

# Enable foreign key constraints
cur.execute("PRAGMA foreign_keys = ON")

# Remove existing tables so the notebook can be run again
cur.execute("DROP TABLE IF EXISTS review")
cur.execute("DROP TABLE IF EXISTS restaurant")

# Create the relational tables
cur.executescript("""
CREATE TABLE restaurant (
    restaurant_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE,
    phone TEXT,
    cuisine TEXT NOT NULL
);

CREATE TABLE review (
    review_id INTEGER PRIMARY KEY,
    restaurant_id INTEGER NOT NULL,
    rating INTEGER CHECK (rating IS NULL OR rating BETWEEN 1 AND 10),
    review_timestamp TEXT NOT NULL,
    FOREIGN KEY (restaurant_id)
        REFERENCES restaurant(restaurant_id)
);

CREATE INDEX idx_review_restaurant
    ON review(restaurant_id);

CREATE INDEX idx_review_rating_time
    ON review(rating, review_timestamp);

CREATE INDEX idx_restaurant_cuisine
    ON restaurant(cuisine);
""")

# Insert the DataFrames into the existing tables
restaurant.to_sql(
    "restaurant",
    con,
    index=False,
    if_exists="append"
)

review.to_sql(
    "review",
    con,
    index=False,
    if_exists="append"
)

# Save changes
con.commit()

print("Database created and data inserted successfully.")

Database created and data inserted successfully.


In [ ]:
# import sqlite3
# con=sqlite3.connect("homework2.sqlite")
# cur=con.cursor(); cur.execute("PRAGMA foreign_keys=ON")
# cur.executescript("""
# CREATE TABLE IF NOT EXISTS restaurant (
#  restaurant_id INTEGER PRIMARY KEY,
#  name TEXT NOT NULL UNIQUE,
#  phone TEXT,
#  cuisine TEXT NOT NULL);
# CREATE TABLE IF NOT EXISTS review (
#  review_id INTEGER PRIMARY KEY,
#  restaurant_id INTEGER NOT NULL,
#  rating INTEGER CHECK (rating IS NULL OR rating BETWEEN 1 AND 10),
#  review_timestamp TEXT NOT NULL,
#  FOREIGN KEY (restaurant_id) REFERENCES restaurant(restaurant_id));
# CREATE INDEX IF NOT EXISTS idx_review_restaurant ON review(restaurant_id);
# CREATE INDEX IF NOT EXISTS idx_review_rating_time ON review(rating,review_timestamp);
# CREATE INDEX IF NOT EXISTS idx_restaurant_cuisine ON restaurant(cuisine);
# """)
# restaurant.to_sql("restaurant",con,index=False,if_exists="append")
# review.to_sql("review",con,index=False,if_exists="append")
# con.commit()

### Required questions

For the date-dependent question, this solution uses the invented period **March 1–31, 2020**.

In [16]:
# 1. Restaurant with the most reviews
q1=pd.read_sql_query("""
SELECT r.name, COUNT(*) AS review_count
FROM restaurant r JOIN review v ON r.restaurant_id=v.restaurant_id
GROUP BY r.restaurant_id,r.name
ORDER BY review_count DESC LIMIT 1;
""",con)

# 2. Restaurants receiving a rating of 10 during March 2020
q2=pd.read_sql_query("""
SELECT DISTINCT r.name
FROM restaurant r JOIN review v ON r.restaurant_id=v.restaurant_id
WHERE v.rating=10
  AND v.review_timestamp BETWEEN '2020-03-01 00:00:00' AND '2020-03-31 23:00:00'
ORDER BY r.name;
""",con)

# 3. Cuisine with highest mean rating
q3=pd.read_sql_query("""
SELECT r.cuisine, AVG(v.rating) AS mean_rating
FROM restaurant r JOIN review v ON r.restaurant_id=v.restaurant_id
WHERE v.rating IS NOT NULL
GROUP BY r.cuisine
ORDER BY mean_rating DESC LIMIT 1;
""",con)
print(q1)
print(q2)
print(q3)

                   name  review_count
0  City City Restaurant           156
                        name
0         Big Big Restaurant
1     Big Delight Restaurant
2             Big Sweet Inc.
3         Big Sweet Take-Out
4         Big Tasty Take-Out
5        City Big Restaurant
6       City City Restaurant
7          City Kitchen Inc.
8            City Salty Inc.
9        City Salty Take-Out
10     City Tasty Restaurant
11    Delight Big Restaurant
12         Delight Goat Inc.
13  Delight Salty Restaurant
14      Fish City Restaurant
15   Fish Delight Restaurant
16         Goat Big Take-Out
17       Goat Salty Take-Out
18           Goat Sweet Inc.
19        Kitchen Tasty Inc.
20        Salty Big Take-Out
21    Salty Delight Take-Out
22       Salty Fish Take-Out
23  Salty Kitchen Restaurant
24    Salty Salty Restaurant
25     Sweet City Restaurant
26    Sweet Delight Take-Out
27  Sweet Kitchen Restaurant
28      Sweet Salty Take-Out
29        Tasty Delight Inc.
30     Tasty Goat Restaur

### Extra Credit - Answer using only the key/value model

The same three questions can be answered from the cleaned key/value representation before loading SQLite. The logic below uses the normalized keys and the same invented timestamps.

In [19]:
kv_review = review.merge(
    restaurant,
    on="restaurant_id",
    how="inner"
)

# EC 1: Restaurant with the most reviews
ec1 = (
    kv_review
    .groupby(["restaurant_id", "name"])
    .size()
    .reset_index(name="review_count")
    .sort_values("review_count", ascending=False)
    .head(1)
)

# EC 2: Restaurants with rating 10 during March 2020
ec2 = (
    kv_review[
        (kv_review["rating"] == 10) &
        (kv_review["review_timestamp"].between(
            "2020-03-01 00:00:00",
            "2020-03-31 23:00:00"
        ))
    ][["name"]]
    .drop_duplicates()
    .sort_values("name")
)

# EC 3: Cuisine with the highest average rating
ec3 = (
    kv_review
    .dropna(subset=["rating"])
    .groupby("cuisine")["rating"]
    .mean()
    .sort_values(ascending=False)
    .head(1)
    .reset_index(name="mean_rating")
)

print("EC 1:")
display(ec1)

print("EC 2:")
display(ec2)

print("EC 3:")
display(ec3)

EC 1:


,restaurant_id,name,review_count
7,8,City City Restaurant,156


EC 2:


,name
1441,Big Big Restaurant
1479,Big Delight Restaurant
1490,Big Sweet Inc.
1696,Big Sweet Take-Out
1960,Big Tasty Take-Out
1455,City Big Restaurant
1858,City City Restaurant
1527,City Kitchen Inc.
1468,City Salty Inc.
1538,City Salty Take-Out


EC 3:


,cuisine,mean_rating
0,Italian,5.584184


In [ ]:
# kv_review=review.merge(restaurant,on="restaurant_id")

# # EC 1
# ec1=(kv_review.groupby(["restaurant_id","restaurant"]).size()
#      .reset_index(name="review_count")
#      .sort_values("review_count",ascending=False).head(1))

# # EC 2
# ec2=(kv_review[(kv_review.rating==10) &
#                  kv_review.review_timestamp.between("2020-03-01 00:00:00","2020-03-31 23:00:00")]
#      [["restaurant"]].drop_duplicates().sort_values("restaurant"))

# # EC 3
# ec3=(kv_review.dropna(subset=["rating"]).groupby("cuisine").rating.mean()
#      .sort_values(ascending=False).head(1).reset_index(name="mean_rating"))
# print(ec1)
# print(ec2)
# print(ec3)

### Key results

- Counties: 3,221
- Raw occupied ratios above 1: 5; final occupied is clipped to [0,1], with occupied_raw retained for diagnosis
- MultiGeometry counties: 79
- Restaurants: 44
- Review rows: 5000
- Missing ratings: 50
- Most reviews: City City Restaurant (156)
- Cuisine with highest mean rating: Italian (5.584184)
- Date period used for Question 2: March 1–31, 2020
- Restaurants with rating 10 during that period: 33

### Final results

- Exercise 1: 3,221 counties. The raw `occupied` ratio has five values above 1 because the supplied `CENSUSAREA` field is inconsistent with the coordinate bounding boxes; these are retained in `occupied_raw` and bounded in the final `occupied` field.
- Extra Credit 1: GeoJSON agrees with the original shapefile to three decimal places; KML differs by approximately 0.001–0.004 degrees in the four extrema.
- Extra Credit 2: 79 counties contain `MultiGeometry`.
- Exercise 2: 44 restaurants and 5,000 review rows, of which 50 have missing ratings. The relational model uses `restaurant_id` as a foreign key in `review`.